# 重現最終提交檔（最後一步）：競賽一 Player（v7x）

AI CUP 2026 圍棋・競賽一 Player（最終上傳 v7x）。本筆記本從保存的中間檔重現**最終提交檔的最後一步**，並印出 sha256 與預期值比對。

> **關於中間檔**：最後一步需要的「通道分數矩陣」（`real_*.npz`）與舊題 800 列保存檔，都是由主辦方的官方資料算出來的，**不放在本 repo，也不放在 Release**。它們只提供給主辦方（私人連結），請先把檔案放到一個資料夾（例如掛載的 Google Drive），並把下面「設定」格的 `INTERMEDIATE_DIR` 指到該資料夾。沒有這些檔，請改看 `full_pipeline.ipynb`（從官方資料重新產生的步驟）。
> 我們自訓的權重則在 GitHub Release，由 `RELEASE_URL` 下載。

| 步驟 | 內容 | 硬體 | 耗時 |
|---|---|---|---|
| 1–2 | clone repo、安裝鎖定版本套件 | CPU | 約 2–5 分鐘 |
| 3 | 外部預訓練權重（預設略過，最後一步用不到） | — | 下載約 1.1 GB（若開） |
| 4 | 從 GitHub Release 下載自訓權重（約 128 MB）並驗 sha256 | CPU | 數十秒 |
| 5 | 從 `INTERMEDIATE_DIR` 讀入最後一步需要的中間檔（約 10 MB）並驗 sha256 | CPU | 數秒 |
| 7 | Player 最後一步 | CPU | 數秒 |
| 8 | sha256 比對 | CPU | 數秒 |

**Colab 免費版就能跑完，不需要 GPU。** 最後一步本身不讀自訓權重（權重是上游步驟用的），第 4 格下載它們只是為了完整性，`WEIGHTS = "none"` 可略過。

這份筆記本**只重現最後一步**。從棋譜整條重跑做不到公開重現的程度（上游中間產物已刪、重編約 70 小時，另外還要重訓三個認人網路），分段說明與每段的 GPU 需求、耗時見 `colab/full_pipeline.ipynb` 與 `PIPELINE.md`。
舊題 800 列（v6）**不是**這裡重算的：它是 11/06 的保存檔，這裡只核對 sha256（原因與缺哪些檔見 `PIPELINE.md`「舊題 800 列」）。

In [ ]:
# 設定。標示「待補」的值要等 GitHub repo 與 Release 建好後填入。
import os, sys, subprocess
from pathlib import Path

REPO_URL    = os.environ.get("REPO_URL", "待補")        # 例：https://github.com/<帳號>/aicup2026-go-player.git
RELEASE_URL = os.environ.get("RELEASE_URL", "待補")     # 我們自訓權重的下載網址，例：https://github.com/<帳號>/aicup2026-go-player/releases/download/<tag>
INTERMEDIATE_DIR = os.environ.get("INTERMEDIATE_DIR", "/content/drive/MyDrive/aicup2026-go-player-intermediate")   # 官方資料衍生的中間檔（只提供給主辦方，見上面說明）
REPO_DIR    = Path(os.environ.get("REPO_DIR", "/content/aicup2026-go-player"))
# （環境變數只是方便本機測試；在 Colab 直接改上面的字串即可。）

def run(cmd, cwd=None, env=None, check=True):
    """執行指令並印出輸出（stdout＋stderr）。"""
    e = dict(os.environ, **(env or {}))
    r = subprocess.run(cmd, cwd=str(cwd or REPO_DIR), env=e, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    print(r.stdout[-6000:] if len(r.stdout) > 6000 else r.stdout)
    if check and r.returncode:
        raise RuntimeError(f"指令失敗（exit {r.returncode}）：{cmd}")
    return r


In [ ]:
# 1) 取得程式碼
if not (REPO_DIR / "colab").exists():
    assert REPO_URL != "待補", "請先把上一格的 REPO_URL 填成實際網址"
    run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], cwd="/")
print("repo:", REPO_DIR, "| commit:", run(["git", "rev-parse", "--short", "HEAD"], check=False).stdout.strip())


In [ ]:
# 2) 安裝鎖定版本的套件（requirements.txt；最後一步只需要 CPU）。
#    Colab 若提示「需重新啟動工作階段」就照做，然後從「設定」那格重跑。
run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"])
import importlib.metadata as md
need = {"numpy": "2.5.4", "scipy": "1.18.1", "pandas": "3.0.6", "scikit-learn": "1.9.1", "lightgbm": "4.7.0"}
bad = {k: (md.version(k), v) for k, v in need.items() if md.version(k) != v}
print("Python", sys.version.split()[0], "| 版本核對：", "全部符合" if not bad else f"不符 {bad}")
assert not bad, "scikit-learn 必須是 1.9.1（Player 的候選池分類頭只在此版本位元重現），其他套件請照 requirements.txt"


In [ ]:
# 3) 外部預訓練權重（KataGo 執行檔／網路、HumanSL .ckpt、Strength Estimator）。
#    本 repo 與 Release 都不散布它們；這一格只從官方網址下載（並驗 sha256）。「最後一步重現」用不到，預設不下載（合計約 1.1 GB）。
#    要重新抽特徵或重新訓練（full_pipeline.ipynb）才需要。來源網址、版本、授權見 README「外部資源」。
DOWNLOAD_EXTERNAL = False
sys.path.insert(0, str(REPO_DIR / "colab"))
import colab_utils as U
if DOWNLOAD_EXTERNAL:
    U.fetch_external(REPO_DIR)
else:
    for k, e in U.EXTERNAL.items():
        print(f"  （略過）{k}: {e['url']}  [{e['bytes']:,} bytes, sha256 {e['sha256'][:12]}…, {e['license']}]")


In [ ]:
# 4) 我們自訓的權重：從 GitHub Release 下載並驗 sha256（清單見 ASSETS.md、colab/assets_manifest.json 的 category == "weights"）。
#    WEIGHTS = "all"：全部自訓權重；"final"：只下載最後一步需要的；"none"：不下載。
WEIGHTS = "all"
if WEIGHTS != "none":
    U.fetch_weights(REPO_DIR, RELEASE_URL, final_only=(WEIGHTS == "final"))


In [ ]:
# 5) 官方資料衍生的中間檔：從 INTERMEDIATE_DIR 讀取（只提供給主辦方；本 repo 與 Release 都沒有）。
#    在 Colab 可把主辦方給的私人連結內容放到自己的 Google Drive，掛載後把 INTERMEDIATE_DIR 指到該資料夾。
if not Path(INTERMEDIATE_DIR).is_dir():
    try:
        from google.colab import drive; drive.mount("/content/drive")      # 只在 Colab 有效
    except Exception as e:
        print("（不在 Colab，或掛載失敗）", e)
U.fetch_intermediate(REPO_DIR, INTERMEDIATE_DIR, final_only=True)      # 只取最後一步需要的；每個檔都驗 sha256


In [ ]:
# 5b)（選用）官方資料：只用來驗證 Player 提交檔格式（1,600 題、候選池成員）。主辦方資料不附在 repo。
#    若要驗證，把 contest_1111 的 8 個檔放到 <repo>/data/contest_1111/（例如從 Google Drive 複製）；沒有就略過，輸出檔照樣產生，只是少了這道驗證。
print("官方資料：", "找到" if (REPO_DIR / "data/contest_1111").exists() else "沒有（Player 格式驗證會略過，不影響產生的 CSV）")

In [ ]:
# 6) 預期的 sha256（原專案上傳檔；2026-11-26 實測）
EXPECT = {
    "player_v7x": "b4ad40c24b5c6e991cc69bb08af060de23b3f68c96e64f5d7d79c36ca4c04643",   # Player 最終上傳檔
    "player_v7w": "21a6fde0813d452d4974be5051cafaf44cb264d6b8c10367c3d4afc7c78262e6",   # Player 前一版（--w-new 0）
    "old_v6_800": "b4eed172f033c7a20431c439216a4678022342765615d79610a1851764e98d10",   # 舊題 800 列（v6，原樣沿用；只能核對，不能重算）
}
OUT_DIR = REPO_DIR / "out"; OUT_DIR.mkdir(exist_ok=True)
ENV = {"PYTHONPATH": "src:scripts:.", "WORK_DIR": str(REPO_DIR / ".scratch")}
def fresh(name):                                   # 腳本不覆寫既有輸出，重跑時先刪
    p = OUT_DIR / name
    if p.exists(): p.unlink()
    return str(p)


In [ ]:
# 7) Player 最後一步（數秒，CPU）：通道分數矩陣＋舊題 800 列 → 1,600 列提交檔
#    輸入：.scratch/e90/final/make/real_v7q_S_*.npz、e112/final/real_X_*.npz、e123/final/real_Xlr_*.npz、e127/final/real_Xp1r_*.npz、e62/old_v6_800.csv
#    輸出：out/player_v7x.csv（--w-new .35，最終）與 out/player_v7w.csv（--w-new 0，前一版，用來交叉驗證舊題＋三通道的部分）
run([sys.executable, "final/make_player_final.py", "--out", fresh("player_v7x.csv")], env=ENV)
run([sys.executable, "final/make_player_final.py", "--out", fresh("player_v7w.csv"), "--w-new", "0"], env=ENV)
# 舊題 800 列是保存檔（無法由公開檔重算）：核對 sha256 與格式
run([sys.executable, "final/check_old_v6.py", ".scratch/e62/old_v6_800.csv"], env=ENV)

In [ ]:
# 比對 sha256
import hashlib
sha = lambda p: hashlib.sha256(Path(p).read_bytes()).hexdigest()
rows = [("Player v7x（最終）",   OUT_DIR / "player_v7x.csv", EXPECT["player_v7x"]),
        ("Player v7w（前一版）", OUT_DIR / "player_v7w.csv", EXPECT["player_v7w"]),
        ("舊題 800 列 v6（保存檔）", REPO_DIR / ".scratch/e62/old_v6_800.csv", EXPECT["old_v6_800"])]
ok = True
for name, p, want in rows:
    got = sha(p); same = got == want; ok &= same
    print(f"{'OK ' if same else 'DIFF'} {name:22s} {got}\n{'':27s}預期 {want}")
print("\n全部位元組相同" if ok else "\n有檔案不同：請看上面各步驟的輸出（版本、sha256 下載驗證、官方資料）")
assert ok
